# Stage 1 (current setup) — transition network + BLR posterior search

Offline Optuna (NSGA-II) search for PoSTR's transition network **and** its neural-linear
posterior, on the current setup:

- **standard randomized DeepSea-5** (per-cell action mapping, `mapping_seed = 0`), rewards
  squashed with `tanh` as in the agent's replay buffer;
- **world hypotheses = BLR weight samples** fitted on the trained network's features (as
  `NeuralLinearHead` does in the agent) — dropout is off in the current agent;
- search space: architecture + learning rate, BLR prior precision `blr_coefficient`,
  likelihood precision `noise_variance`, and `context_length ∈ {1, 2}` (1 = the Markov
  model the planner needs).

Objectives are the July ones (both minimised): `mean_grid_dist` (world-model accuracy) and
`overconfident_error_rate` (posterior miscalibration). Reward-model diagnostics are logged
alongside. The July version of this search is `hpo1_transition_search.ipynb` (branch `main`).

Every trial is logged to MLflow (experiment **"HPO — current setup"**) as it finishes; the
Optuna study is stored in `hpo_artifacts/optuna_postr_current.db` and is resumable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json

import numpy as np
import pandas as pd
import torch
import yaml
import optuna
from matplotlib import pyplot as plt

from notebook_utils import generate_transition_data
from optuna_to_mlflow import MLflowTrialLogger
from TRL.tuning.common import ARTIFACTS_DIR, load_base_config, set_global_seed, get_device
from TRL.tuning import transition_search_blr as tsb

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.options.display.float_format = '{:.5f}'.format

config = load_base_config(env='5', deterministic=True)
DEVICE = get_device(config)
print(f'torch {torch.__version__} | optuna {optuna.__version__} | device {DEVICE}')

In [ ]:
# ── Budget knobs ─────────────────────────────────────────────────────────────
SEED         = 42
MAPPING_SEED = 0      # DeepSea action mapping (shared by every trial)
DATA_STEPS   = 2000   # offline dataset size (transitions), shared by all trials
N_TRIALS     = 200
N_HYPOTHESES = 100    # BLR weight samples per evaluation
MAX_EPOCHS   = 300
PATIENCE     = 15
MIN_DELTA    = 5e-4

STUDY_NAME = 'postr_transition_blr_hpo'
STORAGE    = f"sqlite:///{ARTIFACTS_DIR / 'optuna_postr_current.db'}"
MLFLOW_EXPERIMENT = 'HPO — current setup'
MLFLOW_RUN = 'Stage 1 transition + BLR search (randomized DeepSea)'

In [ ]:
# ── Shared offline datasets, one per context length ─────────────────────────
# Same random-policy action sequence for both (reseeded before each), so trials
# with context_length 1 and 2 see the same transitions.
datasets = {}
for ctx in (1, 2):
    set_global_seed(SEED)
    cfg = json.loads(json.dumps(config)); cfg['transition']['context_length'] = ctx
    episodes, test_transitions, obs_space, possible_actions = generate_transition_data(
        cfg, DATA_STEPS, randomize_actions=True, mapping_seed=MAPPING_SEED)
    datasets[ctx] = {
        'states':      torch.tensor(episodes['states'],      dtype=torch.float32),
        'next_states': torch.tensor(episodes['next_states'], dtype=torch.float32),
        'timesteps':   torch.tensor(episodes['timestep'],    dtype=int),
        'rewards':     torch.tanh(torch.tensor(episodes['rewards'], dtype=torch.float32)).unsqueeze(1),
        'actions':     torch.tensor(episodes['actions'],     dtype=int).unsqueeze(1),
        'test_transitions': test_transitions,
        'obs_space': obs_space,
        'possible_actions': possible_actions,
    }
r = datasets[1]['rewards']
print(f'{DATA_STEPS} transitions | goal transitions: {int((r > 0.5).sum())} | '
      f'test states: {len(datasets[1]["test_transitions"])}')

In [ ]:
def objective(trial):
    params = tsb.suggest_params(trial)
    metrics = tsb.train_and_eval(config, params, datasets, DEVICE, max_epochs=MAX_EPOCHS,
                                 patience=PATIENCE, min_delta=MIN_DELTA,
                                 n_hypotheses=N_HYPOTHESES, seed=SEED + trial.number)
    for key, value in {**params, **{k: v for k, v in metrics.items() if k != 'objectives'}}.items():
        trial.set_user_attr(key, value)
    return metrics['objectives']

In [ ]:
ARTIFACTS_DIR.mkdir(exist_ok=True)
study = optuna.create_study(
    study_name=STUDY_NAME, storage=STORAGE,
    directions=['minimize', 'minimize'],
    sampler=optuna.samplers.NSGAIISampler(seed=SEED),
    load_if_exists=True,
)
mlflow_logger = MLflowTrialLogger(MLFLOW_EXPERIMENT, MLFLOW_RUN, params={
    'data_steps': DATA_STEPS, 'n_trials': N_TRIALS, 'n_hypotheses': N_HYPOTHESES,
    'mapping_seed': MAPPING_SEED, 'env': 'DeepSea-5 randomized', 'posterior': 'BLR'})

done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining = max(0, N_TRIALS - done)
print(f'{done} trials already in the study; running {remaining} more')
if remaining:
    study.optimize(objective, n_trials=remaining, callbacks=[mlflow_logger])
print(f'Pareto-front size: {len(study.best_trials)}')

In [ ]:
# ── Trial table ──────────────────────────────────────────────────────────────
ATTRS = ['hidden_dim', 'heads', 'learning_rate', 'num_encoder_layers', 'forward_expansion',
         'context_length', 'blr_coefficient', 'noise_variance',
         'mean_grid_dist', 'overconfident_error_rate', 'exact_match_accuracy',
         'mean_grid_variance', 'reward_r2', 'goal_r_hat', 'goal_reward_margin',
         'epochs_to_converge', 'n_params']
results_df = pd.DataFrame([
    {'trial': t.number, **{a: t.user_attrs.get(a) for a in ATTRS}}
    for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]).sort_values('mean_grid_dist').reset_index(drop=True)
print(f'{len(results_df)} completed trials')
results_df.head(20)

In [ ]:
# ── Pareto front ─────────────────────────────────────────────────────────────
pareto_numbers = {t.number for t in study.best_trials}
is_pareto = results_df['trial'].isin(pareto_numbers)
fig, ax = plt.subplots(figsize=(10, 6))
for ctx, marker in ((1, 'o'), (2, 's')):
    m = results_df['context_length'] == ctx
    sc = ax.scatter(results_df.loc[m, 'mean_grid_dist'], results_df.loc[m, 'overconfident_error_rate'],
                    c=results_df.loc[m, 'exact_match_accuracy'], cmap='RdYlGn', vmin=0, vmax=1,
                    marker=marker, alpha=0.65, s=60, edgecolors='grey', linewidths=0.3,
                    label=f'context_length {ctx}')
ax.scatter(results_df.loc[is_pareto, 'mean_grid_dist'], results_df.loc[is_pareto, 'overconfident_error_rate'],
           color='black', marker='*', s=250, zorder=5, label=f'Pareto front ({is_pareto.sum()} trials)')
plt.colorbar(sc, ax=ax, label='Exact-match accuracy')
ax.set_xlabel('Mean grid distance  (world-model accuracy, lower = better)')
ax.set_ylabel('Overconfident-error rate  (posterior miscalibration, lower = better)')
ax.set_title('Transition + BLR search on randomized DeepSea-5')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Does the Markov model (context_length 1) lose anything? ──────────────────
results_df.groupby('context_length')[['mean_grid_dist', 'overconfident_error_rate',
    'exact_match_accuracy', 'reward_r2', 'goal_reward_margin']].agg(['median', 'min'])

In [ ]:
# ── Export the Pareto candidates and a recommended configuration ─────────────
# Recommendation: the Pareto trial closest to the ideal point after scaling both
# objectives to [0, 1] over the front.
PARAM_KEYS = ['hidden_dim', 'heads', 'learning_rate', 'num_encoder_layers', 'forward_expansion',
              'context_length', 'blr_coefficient', 'noise_variance']
front = [t for t in study.best_trials]
vals = np.array([t.values for t in front])
span = np.where(vals.max(0) > vals.min(0), vals.max(0) - vals.min(0), 1.0)
best = front[int(np.argmin(np.linalg.norm((vals - vals.min(0)) / span, axis=1)))]
candidates = [{'trial': t.number, 'params': {k: t.user_attrs[k] for k in PARAM_KEYS},
               'mean_grid_dist': t.values[0], 'overconfident_error_rate': t.values[1],
               'reward_r2': t.user_attrs.get('reward_r2'),
               'goal_reward_margin': t.user_attrs.get('goal_reward_margin'),
               'exact_match_accuracy': t.user_attrs.get('exact_match_accuracy')} for t in front]
with open(ARTIFACTS_DIR / 'transition_blr_pareto.json', 'w') as f:
    json.dump(candidates, f, indent=2)
rec = {k: best.user_attrs[k] for k in PARAM_KEYS}
with open(ARTIFACTS_DIR / 'best_transition_blr_config.yaml', 'w') as f:
    yaml.safe_dump({'trial': best.number, 'transition': {k: rec[k] for k in PARAM_KEYS[:6]},
                    'algorithm': {'blr_coefficient': rec['blr_coefficient'],
                                  'noise_variance': rec['noise_variance']}}, f, sort_keys=False)
mlflow_logger.client.log_artifact(mlflow_logger.parent_id, str(ARTIFACTS_DIR / 'transition_blr_pareto.json'))
mlflow_logger.client.log_artifact(mlflow_logger.parent_id, str(ARTIFACTS_DIR / 'best_transition_blr_config.yaml'))
mlflow_logger.client.set_tag(mlflow_logger.parent_id, 'recommended_trial', str(best.number))
mlflow_logger.finish()
print(f'Recommended: trial {best.number}  values {best.values}'); rec